# CSL7360 A2 — Track B (FCN) — Kaggle training notebook, fixed version

**Workflow for this notebook:**
1. Run this whole notebook **on Kaggle** (free GPU) to train and save `weights.pt` to `/kaggle/working/`.
2. Download `weights.pt` from the Kaggle output panel.
3. Run the standalone `predict.py` (written out near the end of this notebook) **on your own laptop, CPU-only, single-threaded** — that is the only timing that counts for `T_CPU` / `CES`.

Fixes applied vs. the first draft:
- **Boundary refinement block** added after the final 256×256 upsample (was plain bilinear with no learned refinement — likely the main drag on Boundary F1).
- **Seeds fixed** (`torch`, `numpy`, `random`) for reproducible training.
- **Early CPU timing sanity check** cell, run *before* you sink hours into training, so you catch a budget blow-out early (note: this is only a rough signal on Kaggle's CPU — the number that matters is from your own laptop).
- **Standalone `predict.py`** written out as its own file (not a `--mode` flag on a multi-purpose script), matching the exact deliverable the assignment asks for.


## 0. Dataset setup (read before running)

Kaggle notebooks don't see your filesystem — datasets must be attached via **Add Data** (right sidebar) and appear read-only under `/kaggle/input/...`.

1. Download `DUTS-TR.zip` and `DUTS-TE.zip` from http://saliencydetection.net/duts/ on your own machine first (the official site has no direct Kaggle mirror you should trust blindly).
2. On Kaggle: **Add Data → Upload → New Dataset**, upload both zips (Kaggle unzips automatically if you upload a .zip as a dataset, or you can unzip first and upload folders). Two options:
   - One dataset containing both `DUTS-TR/` and `DUTS-TE/` as subfolders, **or**
   - Two separate datasets, one per split.
3. After attaching, check the *actual* mounted path in the cell below — Kaggle sometimes nests an extra folder level (e.g. `/kaggle/input/duts/DUTS-TR/DUTS-TR/DUTS-TR-Image/...`). Don't assume; list it.
4. Expected structure inside, either way:
   ```
   DUTS-TR/DUTS-TR-Image/*.jpg
   DUTS-TR/DUTS-TR-Mask/*.png
   DUTS-TE/DUTS-TE-Image/*.jpg
   DUTS-TE/DUTS-TE-Mask/*.png
   ```
   This matches `find_image_mask_pairs()` below, which already looks for `DUTS-TR-Image/DUTS-TR-Mask` and `DUTS-TE-Image/DUTS-TE-Mask` automatically.

Your **own captured photos are not used here at all** — see the note at the very end of this notebook for why.


In [ ]:
# ==============================================================================
# SECTION 1: ENVIRONMENT & SINGLE-THREAD CPU CONFIGURATION
# NOTE: on Kaggle this cell's thread pinning is irrelevant to training (GPU does the
# work), but keep it anyway -- it costs nothing and keeps this notebook consistent
# with the version you'll eventually time on CPU.
# ==============================================================================
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

import random
import time
from typing import List, Tuple

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

cv2.setNumThreads(1)
torch.set_num_threads(1)

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())


In [ ]:
# ==============================================================================
# SECTION 1b: DATASET PATHS -- EDIT THESE AFTER YOU'VE ADDED THE DATA
# ==============================================================================
# First, see what Kaggle actually mounted:
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
    if depth <= 3:
        print(root)

# EDIT these two paths once you see the real structure printed above.
DUTS_TR_DIR = "/kaggle/input/<your-dataset-slug>/DUTS-TR"
DUTS_TE_DIR = "/kaggle/input/<your-dataset-slug>/DUTS-TE"
WEIGHTS_OUT = "/kaggle/working/weights.pt"


In [ ]:
# ==============================================================================
# SECTION 2: DATASET LOADING, RESIZING & AUGMENTATION
# ==============================================================================
def resize_photo_256(img_bgr: np.ndarray) -> np.ndarray:
    """Resize a BGR image to 256x256 with cv2.INTER_AREA per Section 1b of the spec."""
    if img_bgr.shape[0] == 256 and img_bgr.shape[1] == 256:
        return img_bgr
    return cv2.resize(img_bgr, (256, 256), interpolation=cv2.INTER_AREA)


def resize_mask_256(mask_gray: np.ndarray) -> np.ndarray:
    """Resize a grayscale mask to 256x256 with cv2.INTER_NEAREST, threshold at 128."""
    if mask_gray.shape[0] != 256 or mask_gray.shape[1] != 256:
        mask_gray = cv2.resize(mask_gray, (256, 256), interpolation=cv2.INTER_NEAREST)
    return (mask_gray >= 128).astype(np.uint8)


def find_image_mask_pairs(root_dir: str) -> List[Tuple[str, str]]:
    candidates = [
        ("DUTS-TR-Image", "DUTS-TR-Mask"),
        ("DUTS-TE-Image", "DUTS-TE-Mask"),
        ("images_256", "masks_256"),
        ("Image", "Mask"),
        ("images", "masks"),
    ]
    img_dir, mask_dir = None, None
    for idir, mdir in candidates:
        p_img = os.path.join(root_dir, idir)
        p_msk = os.path.join(root_dir, mdir)
        if os.path.isdir(p_img) and os.path.isdir(p_msk):
            img_dir, mask_dir = p_img, p_msk
            break
    if img_dir is None:
        raise FileNotFoundError(f"Could not find standard image/mask subfolders in {root_dir}.")

    mask_map = {}
    for fname in os.listdir(mask_dir):
        stem, ext = os.path.splitext(fname)
        if ext.lower() in (".png", ".jpg", ".jpeg", ".bmp"):
            mask_map[stem] = os.path.join(mask_dir, fname)

    pairs = []
    for fname in sorted(os.listdir(img_dir)):
        stem, ext = os.path.splitext(fname)
        if ext.lower() in (".jpg", ".jpeg", ".png", ".bmp") and stem in mask_map:
            pairs.append((os.path.join(img_dir, fname), mask_map[stem]))
    return pairs


def create_train_val_split(duts_tr_dir: str, val_ratio: float = 0.10, seed: int = 42):
    pairs = find_image_mask_pairs(duts_tr_dir)
    rng = random.Random(seed)
    shuffled = list(pairs)
    rng.shuffle(shuffled)
    val_count = int(len(shuffled) * val_ratio)
    return shuffled[val_count:], shuffled[:val_count]


class SegmentationDataset(Dataset):
    def __init__(self, pairs, augment: bool = False):
        self.pairs = pairs
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        mask_gray = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img_bgr is None:
            raise RuntimeError(f"Failed to read image: {img_path}")
        if mask_gray is None:
            raise RuntimeError(f"Failed to read mask: {mask_path}")

        img_256 = resize_photo_256(img_bgr)
        mask_256 = resize_mask_256(mask_gray)

        if self.augment:
            if random.random() < 0.5:
                img_256 = np.ascontiguousarray(img_256[:, ::-1, :])
                mask_256 = np.ascontiguousarray(mask_256[:, ::-1])
            if random.random() < 0.15:
                img_256 = np.ascontiguousarray(img_256[::-1, :, :])
                mask_256 = np.ascontiguousarray(mask_256[::-1, :])
            if random.random() < 0.5:
                alpha = random.uniform(0.85, 1.15)
                beta = random.uniform(-15.0, 15.0)
                img_256 = np.clip(img_256.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

        img_rgb = cv2.cvtColor(img_256, cv2.COLOR_BGR2RGB)
        img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).float() / 255.0
        mask_tensor = torch.from_numpy(mask_256).unsqueeze(0).float()
        return img_tensor, mask_tensor


In [ ]:
# ==============================================================================
# SECTION 3: EVALUATION METRICS (mIoU global confusion matrix + vectorized BF1)
# ==============================================================================
def extract_boundary_4conn(mask: torch.Tensor) -> torch.Tensor:
    if mask.dim() == 2:
        mask = mask.unsqueeze(0).unsqueeze(0)
    elif mask.dim() == 3:
        mask = mask.unsqueeze(1)
    mask_bool = mask > 0.5
    padded = F.pad(mask_bool.float(), (1, 1, 1, 1), mode="constant", value=0.0) > 0.5
    up = padded[:, :, :-2, 1:-1]
    down = padded[:, :, 2:, 1:-1]
    left = padded[:, :, 1:-1, :-2]
    right = padded[:, :, 1:-1, 2:]
    interior = mask_bool & up & down & left & right
    return (mask_bool & (~interior)).float()


def compute_bf1_single(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    pred_t = torch.from_numpy((pred_mask > 0).astype(np.float32)).unsqueeze(0).unsqueeze(0)
    true_t = torch.from_numpy((true_mask > 0).astype(np.float32)).unsqueeze(0).unsqueeze(0)
    pred_b = extract_boundary_4conn(pred_t)
    true_b = extract_boundary_4conn(true_t)
    num_pred_b = float(pred_b.sum().item())
    num_true_b = float(true_b.sum().item())
    if num_pred_b == 0 and num_true_b == 0:
        return 1.0
    if num_pred_b == 0 or num_true_b == 0:
        return 0.0
    dilated_true_b = F.max_pool2d(true_b, kernel_size=5, stride=1, padding=2)
    dilated_pred_b = F.max_pool2d(pred_b, kernel_size=5, stride=1, padding=2)
    correct_pred = (pred_b * dilated_true_b).sum().item()
    found_true = (true_b * dilated_pred_b).sum().item()
    precision = correct_pred / num_pred_b
    recall = found_true / num_true_b
    if precision + recall == 0.0:
        return 0.0
    return float(2.0 * precision * recall / (precision + recall))


class SegmentationEvaluator:
    def __init__(self):
        self.reset()

    def reset(self):
        self.tn = self.fp = self.fn = self.tp = 0
        self.bf1_scores = []

    def update(self, pred_mask: np.ndarray, true_mask: np.ndarray):
        pred_b = (pred_mask > 0).astype(np.uint8)
        true_b = (true_mask > 0).astype(np.uint8)
        self.tp += int(np.sum((pred_b == 1) & (true_b == 1)))
        self.fp += int(np.sum((pred_b == 1) & (true_b == 0)))
        self.fn += int(np.sum((pred_b == 0) & (true_b == 1)))
        self.tn += int(np.sum((pred_b == 0) & (true_b == 0)))
        self.bf1_scores.append(compute_bf1_single(pred_b, true_b))

    def compute(self) -> dict:
        denom_obj = self.tp + self.fp + self.fn
        denom_bg = self.tn + self.fp + self.fn
        iou_obj = (self.tp / denom_obj) if denom_obj > 0 else 1.0
        iou_bg = (self.tn / denom_bg) if denom_bg > 0 else 1.0
        miou = 0.5 * (iou_obj + iou_bg)
        bf1 = float(np.mean(self.bf1_scores)) if self.bf1_scores else 0.0
        q_score = 0.70 * miou + 0.30 * bf1
        return {"iou_obj": iou_obj, "iou_bg": iou_bg, "mIoU": miou, "BF1": bf1, "Q": q_score}


def compute_ces(q_score: float, t_cpu_ms: float, s_disk_bytes: int) -> float:
    if t_cpu_ms > 50.0:
        return 0.0
    four_mib = 4 * 1024 * 1024
    time_penalty = 0.20 * (t_cpu_ms / 50.0)
    size_penalty = 0.10 * min(1.0, s_disk_bytes / four_mib)
    return float(q_score * (1.0 - time_penalty - size_penalty))


In [ ]:
# ==============================================================================
# SECTION 4: MODEL  --  *** FIX 1: boundary refinement block added at 256x256 ***
# The original version jumped straight from a 128x128 head to a plain bilinear
# resize to 256x256, with no learned layer operating at full resolution. That
# throws away exactly the fine-grained detail Boundary F1 cares about. This
# version adds one cheap depthwise-separable block AFTER the upsample, so the
# network gets one chance to sharpen edges at full resolution before the 1x1
# classifier head. Still entirely depthwise-separable, so the added cost is small.
# ==============================================================================
class DSConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, kernel_size=3, stride=stride, padding=1, groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class FastSegUNet(nn.Module):
    def __init__(self, base_ch: int = 24):
        super().__init__()
        c1, c2, c3, c4, c5 = base_ch, base_ch * 2, base_ch * 3, base_ch * 4, base_ch * 6

        self.stem = nn.Sequential(
            nn.Conv2d(3, c1, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c1),
            nn.ReLU(inplace=True),
            DSConvBlock(c1, c1, stride=1),
        )
        self.enc2 = nn.Sequential(DSConvBlock(c1, c2, stride=2), DSConvBlock(c2, c2, stride=1))
        self.enc3 = nn.Sequential(DSConvBlock(c2, c3, stride=2), DSConvBlock(c3, c3, stride=1))
        self.enc4 = nn.Sequential(DSConvBlock(c3, c4, stride=2), DSConvBlock(c4, c4, stride=1))
        self.bottleneck = nn.Sequential(DSConvBlock(c4, c5, stride=2), DSConvBlock(c5, c4, stride=1))

        self.dec4 = DSConvBlock(c4 + c4, c3, stride=1)
        self.dec3 = DSConvBlock(c3 + c3, c2, stride=1)
        self.dec2 = DSConvBlock(c2 + c2, c1, stride=1)
        self.dec1 = DSConvBlock(c1 + c1, c1, stride=1)

        # *** FIX 1: refinement block at full 256x256 resolution ***
        self.refine_256 = DSConvBlock(c1, c1, stride=1)
        self.head = nn.Conv2d(c1, 1, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        s1 = self.stem(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b = self.bottleneck(s4)

        u4 = F.interpolate(b, size=(16, 16), mode="bilinear", align_corners=False)
        d4 = self.dec4(torch.cat([u4, s4], dim=1))
        u3 = F.interpolate(d4, size=(32, 32), mode="bilinear", align_corners=False)
        d3 = self.dec3(torch.cat([u3, s3], dim=1))
        u2 = F.interpolate(d3, size=(64, 64), mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, s2], dim=1))
        u1 = F.interpolate(d2, size=(128, 128), mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, s1], dim=1))

        u0 = F.interpolate(d1, size=(256, 256), mode="bilinear", align_corners=False)
        r0 = self.refine_256(u0)          # *** FIX 1: learned refinement at full res ***
        logits_256 = self.head(r0)
        return logits_256


@torch.no_grad()
def segment_photo_in_memory(model: nn.Module, img_256_bgr: np.ndarray) -> np.ndarray:
    img_rgb = cv2.cvtColor(img_256_bgr, cv2.COLOR_BGR2RGB)
    x = torch.from_numpy(img_rgb).permute(2, 0, 1).unsqueeze(0).float().mul_(1.0 / 255.0)
    logits = model(x)
    return (logits[0, 0] > 0.0).to(torch.uint8).numpy()


def save_weights_fp16(model: nn.Module, path: str):
    fp16_state = {k: v.detach().cpu().half() for k, v in model.state_dict().items()}
    torch.save(fp16_state, path)


def load_weights_fp16(model: nn.Module, path: str, device: str = "cpu") -> nn.Module:
    state = torch.load(path, map_location=device, weights_only=True)
    fp32_state = {k: v.float() for k, v in state.items()}
    model.load_state_dict(fp32_state)
    model.to(device)
    model.eval()
    return model


In [ ]:
# ==============================================================================
# SECTION 4b: *** FIX 4 -- early CPU timing sanity check ***
# Run this BEFORE committing hours to training, so a budget blow-out is caught
# immediately rather than after the fact. This number is measured on Kaggle's
# CPU, which is NOT your laptop and NOT the number that counts for grading --
# it's only a cheap early warning. You must re-measure on your own laptop later.
# ==============================================================================
_sanity_model = FastSegUNet(base_ch=24)
_sanity_model.eval()
_dummy = torch.zeros(1, 3, 256, 256, dtype=torch.float32)
_traced = torch.jit.trace(_sanity_model, _dummy)
_traced.eval()

_dummy_img = (np.random.rand(256, 256, 3) * 255).astype(np.uint8)
_ = segment_photo_in_memory(_traced, _dummy_img)  # warm-up

_t0 = time.process_time()
for _ in range(50):
    _ = segment_photo_in_memory(_traced, _dummy_img)
_t1 = time.process_time()
print(f"[Kaggle CPU, rough signal only] ~{(_t1 - _t0) / 50 * 1000:.2f} ms/image "
      f"-- budget is 50 ms on YOUR laptop, this is just a sanity check")


In [ ]:
# ==============================================================================
# SECTION 5: LOSS
# ==============================================================================
class HybridSegmentationLoss(nn.Module):
    def __init__(self, bce_weight: float = 0.5, dice_weight: float = 0.5, edge_boost: float = 2.0):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.edge_boost = edge_boost

    def forward(self, logits, targets):
        with torch.no_grad():
            true_b = extract_boundary_4conn(targets)
            edge_band = F.max_pool2d(true_b, kernel_size=5, stride=1, padding=2)
            pixel_weights = 1.0 + self.edge_boost * edge_band

        bce_per_pixel = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        weighted_bce = (bce_per_pixel * pixel_weights).mean()

        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum(dim=(1, 2, 3))
        union = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        soft_dice = 1.0 - ((2.0 * intersection + 1e-6) / (union + 1e-6)).mean()

        return self.bce_weight * weighted_bce + self.dice_weight * soft_dice


def evaluate_dataloader(model, val_loader, device) -> dict:
    model.eval()
    evaluator = SegmentationEvaluator()
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            preds = (logits > 0.0).cpu().numpy()
            trues = masks.numpy()
            for i in range(preds.shape[0]):
                evaluator.update(preds[i, 0], trues[i, 0])
    return evaluator.compute()


In [ ]:
# ==============================================================================
# SECTION 6: TRAINING  --  *** FIX 2: seeds fixed for reproducibility ***
# ==============================================================================
def set_all_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def run_training(duts_tr_dir, weights_out, epochs=35, batch_size=32, lr=1e-3,
                  base_ch=24, val_ratio=0.10, seed=42):
    set_all_seeds(seed)  # *** FIX 2 ***
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    train_pairs, val_pairs = create_train_val_split(duts_tr_dir, val_ratio=val_ratio, seed=seed)
    print(f"DUTS-TR split -> train: {len(train_pairs)} | val: {len(val_pairs)}")

    train_ds = SegmentationDataset(train_pairs, augment=True)
    val_ds = SegmentationDataset(val_pairs, augment=False)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    model = FastSegUNet(base_ch=base_ch).to(device)
    criterion = HybridSegmentationLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    best_q = -1.0
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        t0 = time.time()
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, masks)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
        scheduler.step()
        avg_loss = running_loss / len(train_ds)
        metrics = evaluate_dataloader(model, val_loader, device)
        elapsed = time.time() - t0
        print(f"Epoch [{epoch:02d}/{epochs:02d}] Loss: {avg_loss:.4f} | "
              f"Val mIoU: {metrics['mIoU']:.4f} | Val BF1: {metrics['BF1']:.4f} | "
              f"Val Q: {metrics['Q']:.4f} ({elapsed:.1f}s)")
        if metrics["Q"] > best_q:
            best_q = metrics["Q"]
            save_weights_fp16(model, weights_out)
            print(f"  --> saved new best to '{weights_out}' "
                  f"(Val Q={best_q:.4f}, size={os.path.getsize(weights_out)/1024:.1f} KiB)")
    return model, best_q


In [ ]:
# ==============================================================================
# RUN TRAINING -- make sure DUTS_TR_DIR above is correct before running this
# ==============================================================================
model, best_val_q = run_training(
    duts_tr_dir=DUTS_TR_DIR,
    weights_out=WEIGHTS_OUT,
    epochs=35,
    batch_size=32,
    lr=1e-3,
    base_ch=24,
    val_ratio=0.10,
    seed=42,
)
print("Best validation Q:", best_val_q)
print("Weights saved to:", WEIGHTS_OUT, "| size:", os.path.getsize(WEIGHTS_OUT), "bytes")


## Download the weights

Open the **Output** panel on the right side of the Kaggle notebook (or the
`/kaggle/working/` file browser), find `weights.pt`, and download it to your
laptop. Everything after this point in the assignment — timing, `predict.py`,
CES, the captured-photos evaluation, the report — happens **locally**, not on
Kaggle, because the timing rule in Section 2b is specifically about *your own
laptop's* single-core CPU time.


In [ ]:
# ==============================================================================
# *** FIX 3: standalone predict.py, written as its own file ***
# This is the literal deliverable: `python predict.py --image photo.jpg --out mask.png`
# Run this cell once to produce predict.py, then copy it (and weights.pt) to your laptop.
# ==============================================================================
predict_py_source = r'''
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

import argparse
import time

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

cv2.setNumThreads(1)
torch.set_num_threads(1)


def resize_photo_256(img_bgr):
    if img_bgr.shape[0] == 256 and img_bgr.shape[1] == 256:
        return img_bgr
    return cv2.resize(img_bgr, (256, 256), interpolation=cv2.INTER_AREA)


class DSConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, kernel_size=3, stride=stride, padding=1, groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class FastSegUNet(nn.Module):
    def __init__(self, base_ch=24):
        super().__init__()
        c1, c2, c3, c4, c5 = base_ch, base_ch * 2, base_ch * 3, base_ch * 4, base_ch * 6
        self.stem = nn.Sequential(
            nn.Conv2d(3, c1, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
            DSConvBlock(c1, c1, stride=1),
        )
        self.enc2 = nn.Sequential(DSConvBlock(c1, c2, stride=2), DSConvBlock(c2, c2, stride=1))
        self.enc3 = nn.Sequential(DSConvBlock(c2, c3, stride=2), DSConvBlock(c3, c3, stride=1))
        self.enc4 = nn.Sequential(DSConvBlock(c3, c4, stride=2), DSConvBlock(c4, c4, stride=1))
        self.bottleneck = nn.Sequential(DSConvBlock(c4, c5, stride=2), DSConvBlock(c5, c4, stride=1))
        self.dec4 = DSConvBlock(c4 + c4, c3, stride=1)
        self.dec3 = DSConvBlock(c3 + c3, c2, stride=1)
        self.dec2 = DSConvBlock(c2 + c2, c1, stride=1)
        self.dec1 = DSConvBlock(c1 + c1, c1, stride=1)
        self.refine_256 = DSConvBlock(c1, c1, stride=1)
        self.head = nn.Conv2d(c1, 1, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        s1 = self.stem(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b = self.bottleneck(s4)
        u4 = F.interpolate(b, size=(16, 16), mode="bilinear", align_corners=False)
        d4 = self.dec4(torch.cat([u4, s4], dim=1))
        u3 = F.interpolate(d4, size=(32, 32), mode="bilinear", align_corners=False)
        d3 = self.dec3(torch.cat([u3, s3], dim=1))
        u2 = F.interpolate(d3, size=(64, 64), mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, s2], dim=1))
        u1 = F.interpolate(d2, size=(128, 128), mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, s1], dim=1))
        u0 = F.interpolate(d1, size=(256, 256), mode="bilinear", align_corners=False)
        r0 = self.refine_256(u0)
        return self.head(r0)


@torch.no_grad()
def segment_photo_in_memory(model, img_256_bgr):
    img_rgb = cv2.cvtColor(img_256_bgr, cv2.COLOR_BGR2RGB)
    x = torch.from_numpy(img_rgb).permute(2, 0, 1).unsqueeze(0).float().mul_(1.0 / 255.0)
    logits = model(x)
    return (logits[0, 0] > 0.0).to(torch.uint8).numpy()


def load_weights_fp16(model, path, device="cpu"):
    state = torch.load(path, map_location=device, weights_only=True)
    model.load_state_dict({k: v.float() for k, v in state.items()})
    model.to(device)
    model.eval()
    return model


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image", type=str, required=True)
    parser.add_argument("--out", type=str, required=True)
    parser.add_argument("--weights", type=str, default="weights.pt")
    parser.add_argument("--base_ch", type=int, default=24)
    args = parser.parse_args()

    s_disk_bytes = os.path.getsize(args.weights)

    model = FastSegUNet(base_ch=args.base_ch)
    load_weights_fp16(model, args.weights, device="cpu")
    dummy = torch.zeros(1, 3, 256, 256, dtype=torch.float32)
    traced_model = torch.jit.trace(model, dummy)
    traced_model.eval()

    img_bgr = cv2.imread(args.image, cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise RuntimeError(f"Could not read image from {args.image}")
    img_256 = resize_photo_256(img_bgr)

    mask_01 = segment_photo_in_memory(traced_model, img_256)  # first run: warm-up + output
    cv2.imwrite(args.out, (mask_01 * 255).astype(np.uint8))

    num_runs = 20
    t_start = time.process_time()
    for _ in range(num_runs):
        _ = segment_photo_in_memory(traced_model, img_256)
    t_end = time.process_time()
    avg_cpu_ms = ((t_end - t_start) / num_runs) * 1000.0

    print(f"T_CPU_ms: {avg_cpu_ms:.2f}")
    print(f"S_disk_bytes: {s_disk_bytes}")


if __name__ == "__main__":
    main()
'''

with open("/kaggle/working/predict.py", "w") as f:
    f.write(predict_py_source)
print("Wrote /kaggle/working/predict.py -- download this alongside weights.pt")


## What else you need besides the DUTS dataset

The dataset + this notebook only gets you the trained model. The assignment deliverables
also require, all done **outside** Kaggle:

1. **Your own captured photos** (5–10 phone photos + hand-drawn masks) — Section 1c.
   These are **never used in training or validation here** — the spec explicitly says
   *"Use these photos only for testing."* Where they come in:
   - A `captured/` folder with `original/`, `images_256/`, `masks_256/` subfolders — you
     create this locally with your phone + an editor (Paint/GIMP/LabelMe), not with code.
   - After training, you run the **local** evaluation (same `SegmentationEvaluator` /
     `run_single_core_eval` logic from the full script) over `captured/` to get a
     mIoU/BF1/Q row for the report — this row is *not* part of CES (CES is DUTS-TE only),
     it's a generalization check.
   - The side-by-side overlay images for every captured photo, for the report.
   - Material for the "discuss at least two failure cases" section — these are the
     photos most likely to expose weaknesses (clutter, occlusion, odd lighting) since
     they weren't drawn from DUTS at all.

2. **Your own laptop**, CPU-only, single thread — this is where `predict.py` and the
   full `T_CPU` measurement over all 5,019 DUTS-TE images must actually run. Kaggle's
   CPU number means nothing for grading.

3. **The saved weights file** (`weights.pt`) and **`predict.py`**, both downloaded from
   Kaggle to sit next to each other locally.

4. **The written report**: CPU model/spec, the required results table (val / DUTS-TE /
   captured, each with mIoU, BF1, Q, plus T_CPU, S_disk, CES), the captured-photo overlay
   figures, and the failure-case discussion — none of this is generated by code, it's
   written by you from the numbers and images the pipeline produces.
